In [ ]:
# === 环境初始化（首次运行需 1-2 分钟，之后浏览器缓存）===
import sys
if "pyodide" in sys.modules:
    import micropip
    await micropip.install("torch")
import torch
print(f"PyTorch 已就绪，版本: {torch.__version__}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
class SimpleAgrNet(nn.Module):
    def __init__(self):
        super(SimpleAgrNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32*7*7, 15)
    def forward(self, x):
        x = self.pool(x)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [ ]:
def get_dummy_data(batch_size=32, num_batches=10):
    for _ in range(num_batches):
        images = torch.randn(batch_size, 3, 56, 56)
        labels = torch.randint(0, 4, (batch_size,))
        yield images, labels

In [ ]:
model = SimpleAgrNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

In [ ]:
for epoch in range(3):
    running_loss = 0.0
    for images, labels in get_dummy_data():
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    avg_loss = running_loss/10
    current_lr = scheduler.get_last_lr()[0]
    print(f"优化后模型 - Epoch {epoch+1}/3, 平均损失 = {avg_loss:.4f}, 当前学习率 = {current_lr:.6f}")